In [1]:
import tensorflow as tf
import re
import numpy as np

# 1. Dummy Dataset based on the assignment scenario
questions = ["hello", "how are you"]
answers = ["hi, how are you", "i am fine"]

# Text cleaning function
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"([?.!,])", r" \1 ", text)
    text = re.sub(r'[" "]+', " ", text)
    text = re.sub(r"[^a-zA-Z?.!,]+", " ", text)
    return text

# Clean and add start/end tokens
def preprocess_sentences(sentences):
    return ['<start> ' + clean_text(s) + ' <end>' for s in sentences]

input_texts = preprocess_sentences(questions)
target_texts = preprocess_sentences(answers)

# Tokenization and Vocabulary generation
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tokenizer.fit_on_texts(input_texts + target_texts)

input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

# Pad sequences
max_len_input = max(len(seq) for seq in input_sequences)
max_len_target = max(len(seq) for seq in target_sequences)

input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_sequences, maxlen=max_len_input, padding='post')
target_tensor = tf.keras.preprocessing.sequence.pad_sequences(target_sequences, maxlen=max_len_target, padding='post')

vocab_size = len(tokenizer.word_index) + 1

In [2]:
embedding_dim = 256
units = 512

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.enc_units, return_sequences=True, return_state=True)

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        return output, state_h, state_c

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        # query hidden state shape == (batch_size, hidden size)
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.dec_units, return_sequences=True, return_state=True)
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state_h, state_c, attention_weights

encoder = Encoder(vocab_size, embedding_dim, units)
decoder = Decoder(vocab_size, embedding_dim, units)

In [4]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_val = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_val.dtype)
    loss_val *= mask
    return tf.reduce_mean(loss_val)

@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_h, enc_c = encoder(inp, enc_hidden)
        dec_hidden = enc_h
        dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * inp.shape[0], 1)

        # Teacher forcing - feeding the target as the next input
        for t in range(1, targ.shape[1]):
            predictions, dec_h, dec_c, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)
            dec_hidden = dec_h

    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

# Training Loop
EPOCHS = 100
for epoch in range(EPOCHS):
    enc_hidden = [tf.zeros((len(input_tensor), units)), tf.zeros((len(input_tensor), units))]
    loss = train_step(input_tensor, target_tensor, enc_hidden)
    if epoch % 20 == 0:
        print(f'Epoch {epoch} Loss {loss.numpy():.4f}')

Epoch 0 Loss 1.7760
Epoch 20 Loss 0.7346
Epoch 40 Loss 0.0109
Epoch 60 Loss 0.0010
Epoch 80 Loss 0.0005


In [5]:
def evaluate(sentence):
    sentence = '<start> ' + clean_text(sentence) + ' <end>'
    inputs = [tokenizer.word_index.get(i, 0) for i in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_len_input, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    enc_hidden = [tf.zeros((1, units)), tf.zeros((1, units))]
    enc_out, enc_h, enc_c = encoder(inputs, enc_hidden)

    dec_hidden = enc_h
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)

    attention_plot = np.zeros((max_len_target, max_len_input))

    for t in range(max_len_target):
        predictions, dec_h, dec_c, attention_weights = decoder(dec_input, dec_hidden, enc_out)

        # Store attention weights
        attention_weights = tf.reshape(attention_weights, (-1, ))
        attention_plot[t] = attention_weights.numpy()

        predicted_id = tf.argmax(predictions[0]).numpy()
        result += tokenizer.index_word.get(predicted_id, '') + ' '

        if tokenizer.index_word.get(predicted_id) == '<end>':
            return result.strip(), attention_plot

        dec_input = tf.expand_dims([predicted_id], 0)
        dec_hidden = dec_h

    return result.strip(), attention_plot

# Test the model with the expected output case
test_sentence = "how are you"
response, attention_weights = evaluate(test_sentence)

print(f'\nInput: {test_sentence}')
print(f'Output: {response.replace("<end>", "").strip()}')
# You can use the `attention_weights` matrix here to visualize how the model focuses on "you".


Input: how are you
Output: i am fine
